### Loss Function in Vanilla GANs

In a **vanilla GAN** (Generative Adversarial Network), there are two competing neural networks:

1. The **generator** $ G $, which generates fake data samples.
2. The **discriminator** $ D $, which distinguishes real data samples from fake ones.

The goal of GANs is to train the generator to produce data that is indistinguishable from real data. This competition between the generator and discriminator is formalized in their respective **loss functions**, which are derived from a **min-max game**.

#### Discriminator Loss

The discriminator $ D $ is trained to maximize the probability of correctly classifying real and fake samples. Its loss function $ \mathcal{L}_D $ is given by:

$$
\mathcal{L}_D = -\mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}[\log(D(\mathbf{x}))] - \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(1 - D(G(\mathbf{z})))]
$$

where:

- $ \mathbf{x} \sim p_{\text{data}} $ represents samples from the real data distribution.
- $ \mathbf{z} \sim p_{\mathbf{z}} $ is a noise vector sampled from a prior distribution (e.g., Gaussian), which the generator uses as input.
- $ G(\mathbf{z}) $ is the generated (fake) data sample created by the generator from noise $ \mathbf{z} $.
- $ D(\mathbf{x}) $ is the  discriminator's estimate of the probability that the sample $ \mathbf{x} $ is real.
- $ D(G(\mathbf{z})) $ is the discriminator’s estimate of the probability that the generated sample $ G(\mathbf{z}) $ is real. During discriminator training, we want to minimize $ D(G(\mathbf{z})) $, which is equivalent to maximizing $ 1- D(G(\mathbf{z})) $. During generator training, the goal is reversed: we want this probability to increase.
- In the loss function of a GAN, the E term is the expectation over the data distribution. It can be thought of as computing an average loss across all the data points in a batch or dataset. The expectation 
E means we are averaging over multiple samples from either the real data distribution or the generator’s noise distribution. In practice, this is done using mini-batches of data during training.
  
In practice, the discriminator's loss consists of two main terms:

1. **Real Data Loss**: $ \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}[\log(D(\mathbf{x}))] $ – the expected log-probability that real data samples are classified as real. We want to maximize this.
2. **Fake Data Loss**: $ \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(1 - D(G(\mathbf{z})))] $ – the expected log-probability that fake data samples are classified as fake. We again want to maximize this.

The discriminator's objective is to **maximize** the sum of real data loss and fake data loss which is equivalent to minimizing the negative of this sum. This is encouraging $ D $ to assign high probabilities to real samples and low probabilities to fake ones.

#### Generator Loss

The generator wants the discriminator to classify its generated samples as **real**. Let $p = D(G(\mathbf{z}))$ be the discriminator's estimated probability that a generated sample is real.

- **When training the discriminator:** we want $p$ close to 0, because the generated sample is fake.
- **When training the generator:** we want $p$ close to 1, because the generator wants to fool the discriminator.

During a generator update, the discriminator's parameters are held fixed, but gradients flow through the discriminator to update the generator. The generator changes the samples it produces, which changes the discriminator's output.

There are **two different generator loss functions** below. Both encourage $p$ to increase, but they have different gradients and therefore produce different training updates.

##### 1. Original minimax generator loss

The original GAN minimax game gives the generator this loss to **minimize**:

$$
\mathcal{L}_G^{\text{minimax}} = \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(1 - D(G(\mathbf{z})))].
$$

For one generated sample, this is $\log(1-p)$. As $p$ increases toward 1, $1-p$ approaches 0 and $\log(1-p)$ becomes more negative. **Minimizing this loss therefore encourages the discriminator's probability of "real" to increase.** A loss does not have to be positive: minimizing means moving toward smaller values.

Notice the sign: the discriminator minimizes $-\log(1-p)$ for fake samples, whereas the original generator minimizes $+\log(1-p)$. They have opposing goals.

##### 2. Non-saturating generator loss

In practice, a common replacement is the **non-saturating loss**, which the generator also **minimizes**:

$$
\mathcal{L}_G^{\text{non-saturating}} = -\mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(D(G(\mathbf{z})))].
$$

For one generated sample, this is $-\log(p)$. As $p$ increases toward 1, this loss decreases toward 0. Minimizing $-\log(p)$ is exactly equivalent to maximizing $\log(p)$.

However, $-\log(p)$ and $\log(1-p)$ are **not the same loss**, nor do they give the same training updates. They simply encourage the same direction of change in $p$.

##### Numerical comparison

The following values use natural logarithms and show the loss for one generated sample:

| $p = D(G(\mathbf{z}))$: estimated probability of real | Original minimax loss $\log(1-p)$ | Non-saturating loss $-\log(p)$ |
| --- | ---: | ---: |
| 0.01 | -0.010 | 4.605 |
| 0.50 | -0.693 | 0.693 |
| 0.99 | -4.605 | 0.010 |

Reading downward, the generator fools the discriminator more successfully, and **both losses decrease**. Their absolute values need not match; what matters is how each loss changes and what gradient it supplies to the generator. The expectation in the formulas averages these per-sample losses over noise samples, approximated using a mini-batch during training.

##### Why prefer the non-saturating loss?

Early in training, the discriminator may confidently reject generated samples, so $p \approx 0$. The original minimax loss can then provide a very weak gradient to the generator.

To see why, suppose the discriminator uses a sigmoid output $p = \sigma(a)$, where $a$ is its output logit. The gradients with respect to that logit are:

$$
\frac{\partial \log(1-p)}{\partial a} = -p,
\qquad
\frac{\partial [-\log(p)]}{\partial a} = p-1.
$$

At $p = 0.01$, these gradients are $-0.01$ and $-0.99$, respectively. The original loss supplies a much weaker signal at this point. The non-saturating loss avoids this particular source of vanishing gradients and often helps the generator learn early in training. The full gradient to the generator also depends on the discriminator and generator derivatives; this change alone does not guarantee stable training.

##### Connection to binary cross-entropy

Binary cross-entropy for a predicted probability $p$ and target $y$ is:

$$
\operatorname{BCE}(p,y) = -[y\log(p) + (1-y)\log(1-p)].
$$

During **generator training**, we use the desired target $y=1$:

$$
\operatorname{BCE}(D(G(\mathbf{z})),1) = -\log(D(G(\mathbf{z}))).
$$

Thus, `nn.BCELoss` with target 1 implements the **non-saturating generator loss** after averaging over the batch. The target 1 expresses how the generator wants its samples to be classified; it does not mean those samples are actually real. During discriminator training, generated samples instead receive target 0.

#### Min-Max Game Formulation

In a GAN, the generator and discriminator are playing a **min-max game** where:

$$
\min_G \max_D \mathcal{L}(D, G) = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}[\log(D(\mathbf{x}))] + \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(1 - D(G(\mathbf{z})))]
$$

In this **original minimax formulation**, the generator minimizes this objective, while the discriminator maximizes it (equivalently, minimizes its negative). During a generator update, the real-data term does not depend on the generator, so only the $\log(1-D(G(\mathbf{z})))$ term contributes to its gradient.

When using the **non-saturating generator loss**, we replace the generator's objective with $-\mathbb{E}[\log D(G(\mathbf{z}))]$ while keeping the discriminator objective above. This is a modification of the original minimax training rule, not an algebraic rewriting of the same generator loss.

#### Why This Loss Function?

This adversarial loss function forces the generator to produce outputs that the discriminator cannot distinguish from real data. As the generator improves, the discriminator’s task becomes more challenging, leading to more realistic generated samples. The adversarial nature of this loss function is what drives GANs to produce high-quality outputs.
